# Experiment 001: Baseline CNN

## Objective:

Train and evaluate a reporducible CNN baseline for binary ship classification using the ShipsNet dataset.

## Methodology

- Scene-aware train/validation/test splitting
- Stratified class distribution
- PyTorch CNN baseline
- Validation F1 used for checkpint selection
- Final test set reserved for independent evaluation

## EXP-001 Results

- Baseline CNN test F1: 0.8339, accuracy: 92.29%, precision: 87.60%, recall: 79.58%

In [47]:
#Standard library
import os
import random
from pathlib import Path
import json
import csv

#Data and visualization.
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

#PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset

#Data splitting
from sklearn.model_selection import StratifiedGroupKFold

In [ ]:
default_data_dir = Path.home() / "data" / "shipsnet"

data_dir = Path(
    os.environ.get("SHIPSNET_DATA_DIR", default_data_dir)
)

if not data_dir.exists():
    raise FileNotFoundError(
        f"ShipsNet data directory not found: {data_dir}"
    )

print(f"ShipsNet data directory: {data_dir.resolve()}")

In [ ]:
print(f"PyTorch version: {torch.__version__}")

cuda_available = torch.cuda.is_available()
device = torch.device("cuda" if cuda_available else "cpu")

print(f"CUDA: available: {cuda_available}")
if cuda_available:
     print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"Device: {device}")

In [ ]:
run_training = False
run_test = False


seed = 42
batch_size = 32
learning_rate = 0.01
num_epochs = 20

checkpoint_path = Path("../checkpoints/baseline_best.pt")

experiment_id = "EXP-001-baseline"
overwrite_experiment = False

experiment_dir = Path("../experiments") / experiment_id
experiment_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
random.seed(seed)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

In [ ]:
class ShipsNetDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = Path(data_dir)
        self.image_paths = sorted(self.data_dir.glob("*.png"))

        if not self.image_paths:
            raise FileNotFoundError(
                f"No PNG images found in {self.data_dir}"
            )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = self.image_paths[index]

        image = Image.open(image_path).convert("RGB")
        image_array = np.array(image)

        image_tensor = torch.from_numpy(image_array)
        image_tensor = image_tensor.permute(2, 0, 1)
        image_tensor = image_tensor.float() / 255.0

        label = int(image_path.name.split("__")[0])
        label_tensor = torch.tensor(label, dtype=torch.long)

        return image_tensor, label_tensor


In [ ]:
dataset = ShipsNetDataset(data_dir)
print(f"Dataset size: {len(dataset)}")

assert len(dataset) == 4000, (
    f"Expected 4000 images, found {len(dataset)}"
)

In [ ]:
labels = []
groups = []

for image_path in dataset.image_paths:
    parts = image_path.name.split("__")

    label = int(parts[0])
    scene_id = parts[1]

    labels.append(label)
    groups.append(scene_id)

print(f"Total images: {len(labels)}")
print(f"Unique scenes: {len(set(groups))}")
print(f"Ship images: {sum(labels)}")
print(f"No-ship images: {len(labels) - sum(labels)}")

assert len(labels) == 4000
assert len(set(groups)) == 434
assert sum(labels) == 1000

In [ ]:
sgkf = StratifiedGroupKFold(
    n_splits=20,
    shuffle=True,
    random_state=seed
)

In [ ]:
folds = []

dummy_x = np.zeros(len(labels))

for _, fold_indices in sgkf.split(
    dummy_x,
    labels,
    groups
):
    folds.append(fold_indices)

train_indices = np.concatenate(folds[:14])
val_indices = np.concatenate(folds[14:17])
test_indices = np.concatenate(folds[17:20])

In [ ]:
total_split_images = (
    len(train_indices)
    + len(val_indices)
    + len(test_indices)
)

assert total_split_images == len(dataset), (
    f"Split contains {total_split_images} images, "
    f"expected {len(dataset)}"
)

print(
    f"Split sizes - "
    f"Train: {len(train_indices) / len(dataset):.2%}, "
    f"Validation: {len(val_indices) / len(dataset):.2%}, "
    f"Test: {len(test_indices) / len(dataset):.2%}"
)

In [ ]:
labels_array = np.array(labels)

for name, indices in [
    ("Train", train_indices),
    ("Validation", val_indices),
    ("Test", test_indices),
]:
    split_labels = labels_array[indices]

    ship_count = split_labels.sum()
    total_count = len(split_labels)
    no_ship_count = total_count - ship_count
    ship_rate = ship_count / total_count

    print(
        f"{name}: "
        f"{total_count} images, "
        f"{ship_count} ships, "
        f"{no_ship_count} no-ships, "
        f"{ship_rate:.2%} ships"
    )

In [ ]:
groups_array = np.array(groups)

train_groups = set(groups_array[train_indices])
val_groups = set(groups_array[val_indices])
test_groups = set(groups_array[test_indices])

train_val_overlap = train_groups & val_groups
train_test_overlap = train_groups & test_groups
val_test_overlap = val_groups & test_groups

assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0

print("Scene overlap check passed.")

In [ ]:
train_dataset = Subset(dataset, train_indices.tolist())
val_dataset = Subset(dataset, val_indices.tolist())
test_dataset = Subset(dataset, test_indices.tolist())

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
def create_model():
    return nn.Sequential(
        nn.Conv2d(
            in_channels=3,
            out_channels=8,
            kernel_size=3,
            padding=1
        ),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(
            in_channels=8,
            out_channels=16,
            kernel_size=3,
            padding=1
        ),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Flatten(),

        nn.Linear(
            in_features=16 * 20 * 20,
            out_features=2
        )

    )

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    true_positive = 0
    true_negative = 0
    false_positive = 0
    false_negative = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

            true_positive += (
                (predictions == 1) & (labels == 1)
            ).sum().item()

            true_negative += (
                (predictions == 0) & (labels == 0)
            ).sum().item()

            false_positive += (
                (predictions == 1) & (labels == 0)
            ).sum().item()

            false_negative += (
                (predictions == 0) & (labels == 1)
            ).sum().item()
    
    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    precision_denominator = true_positive + false_positive
    precision = (
        true_positive / precision_denominator
        if precision_denominator > 0
        else 0.0
    )
    recall_denominator = true_positive + false_negative

    recall = (
        true_positive / recall_denominator
        if recall_denominator > 0
        else 0.0
    )

    f1_denominator = precision + recall

    f1 = (
        2 * (precision * recall) / f1_denominator
        if f1_denominator > 0
        else 0.0
    )

    return (
        epoch_loss, 
        epoch_accuracy, 
        precision, 
        recall, 
        f1,
        true_positive,
        true_negative,
        false_positive,
        false_negative
    )

## Training

The following cells initilize and train a new baseline model. Running them starts a fresh training experiment.

In [ ]:
if run_training:
    model = create_model().to(device)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=learning_rate
    )
    
    train_losses = []
    train_accuracies = []

    val_losses = []
    val_accuracies = []
    val_precisions = []
    val_recalls = []
    val_f1_scores = []

In [ ]:
if run_training:
    for epoch in range(num_epochs):
        train_loss, train_accuracy = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        (
            val_loss,
            val_accuracy,
            val_precision,
            val_recall,
            val_f1,
            _,
            _,
            _,
            _
        ) = evaluate(
            model,
            val_loader,
            criterion,
            device
        )

        train_losses.append(train_loss)
        train_accuracies.append(train_accuracy)

        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)
        val_precisions.append(val_precision)
        val_recalls.append(val_recall)
        val_f1_scores.append(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch + 1

            torch.save(
                {
                    "epoch": best_epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": val_loss,
                    "val_accuracy": val_accuracy,
                    "val_precision": val_precision,
                    "val_recall": val_recall,
                    "val_f1": val_f1,
                },
                checkpoint_path
            )

            print(
                f"Saved new best model at epoch {best_epoch} "
                f"(F1: {best_val_f1:.4f})"
            )

        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Train loss: {train_loss:.4f} | "
            f"Train acc: {train_accuracy:.2%} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val acc: {val_accuracy:.2%} | "
            f"Precision: {val_precision:.2%} | "
            f"Recall: {val_recall:.2%} | "
            f"F1: {val_f1:.4f}"
        )

In [ ]:
if not checkpoint_path.exists():
    raise FileNotFoundError(
        F"Checkpoint not found: {checkpoint_path.resolve()}"
    )

best_model = create_model().to(device)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

best_model.load_state_dict(
    checkpoint["model_state_dict"]
)

best_model.eval()


print(f"Best epoch: {checkpoint['epoch']}")
print(f"Best validation F1: {checkpoint['val_f1']:.4f}")
print(f"Validation accuracy: {checkpoint['val_accuracy']:.2%}")
print(f"Precision: {checkpoint['val_precision']:.2%}")
print(f"Recall: {checkpoint['val_recall']:.2%}")

## Final Test Evaluation

The final checkpoint selected using validation F1 is evaluated once on the held-out test set. The test set was not used during model training or model selection.

In [ ]:
if run_test:
    (
        test_loss,
        test_accuracy,
        test_precision,
        test_recall,
        test_f1,
        test_tp,
        test_tn,
        test_fp,
        test_fn
    ) = evaluate(
        best_model,
        test_loader,
        criterion,
        device
    )

    print(f"Test loss: {test_loss:.4f}")
    print(f"Test accuracy: {test_accuracy:.2%}")
    print(f"Test precision: {test_precision:.2%}")
    print(f"Test recall: {test_recall:.2%}")
    print(f"Test F1: {test_f1:.4f}")

    print("\nConfusion matrix counts:")
    print(f"True positives: {test_tp}")
    print(f"True negatives: {test_tn}")
    print(f"False positives: {test_fp}")
    print(f"False negatives: {test_fn}")

In [ ]:
def summarize_split(indices):
    split_labels = labels_array[indices]

    total = len(indices)
    ships = int(split_labels.sum())
    no_ships = total - ships

    return {
        "samples": total,
        "ships": ships,
        "no_ships": no_ships,
        "ship_rate": ships / total
    }

In [ ]:
if run_test:
    experiement_results = {
        "experiment_id": experiment_id,

        "best_checkpoint": {
            "epoch": int(checkpoint["epoch"]),
            "validation": {
                "loss": float(checkpoint["val_loss"]),
                "accuracy": float(checkpoint["val_accuracy"]),
                "precision": float(checkpoint["val_precision"]),
                "recall": float(checkpoint["val_recall"]),
                "f1": float(checkpoint["val_f1"])
            }
        },

        "test": {
            "loss": float(test_loss),
            "accuracy": float(test_accuracy),
            "precision": float(test_precision),
            "recall": float(test_recall),
            "f1": float(test_f1),

            "confusion_matrix": {
                "true_positive": int(test_tp),
                "true_negative": int(test_tn),
                "false_positive": int(test_fp),
                "false_negative": int(test_fn)
            }
        },

        "actual_split": {
            "train": summarize_split(train_indices),
            "validation": summarize_split(val_indices),
            "test": summarize_split(test_indices)
        }
    }

    results_path = experiment_dir / "results.json"

    if results_path.exists() and not overwrite_experiment:
        print(f"Results already exists: {results_path}")
        print("Set overwrite_experiment=True to replace them")
    else:
        with results_path.open("w") as file:
            json.dump(
                experiement_results,
                file,
                indent=4
            )
        print(f"Saved experiment results to: {results_path}")

In [ ]:
experiment_config = {
    "experiment_id": experiment_id,
    "dataset":"ShipsNet",
    "seed": seed,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "num_epochs": num_epochs,
    "optimizer": "SGD",
    "loss_function": "CrossEntropyLoss",
    "split_method": "StratifiedGroupKFold",
    "num_folds": 20,
    "target_split":{
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15
    },
    "checkpoint_metric": "validation_f1"
}

config_path = experiment_dir / "config.jason"

if config_path.exists() and not overwrite_experiment:
    print(f"Config already exists: {config_path}")
    print("Set overwrite_experiment=True to replace it")
else:
    with config_path.open("w") as file:
        json.dump(
            experiment_config,
            file, 
            indent=4
        )
    print(f"Saved experiment configuration to: {config_path}")